In [ ]:
from torch._higher_order_ops import out_dtype
### 파이토치 기반 심층 신경망 구현


#torchvision으로 패션 MNIST 데이터셋 호출
from torchvision.datasets import FashionMNIST

fm_train = FashionMNIST(root='.', train=True, download=True)
fm_test = FashionMNIST(root='.', train=False, download=True)

# fm_train과 fm_test의 속성 => 파이토치 텐서
type(fm_train.data)

print("파이토치 훈련-테스트 입력 크기", fm_train.data.shape, fm_test.data.shape)
print("파이토치 훈련-테스트 타깃 크기", fm_train.targets.shape, fm_test.targets.shape)

# 데이터 전처리
train_input = fm_train.data
train_target = fm_train.targets

train_scaled = train_input / 255.0

from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

print("훈련 세트와 검증 세트 크기: ", train_scaled.shape, val_scaled.shape)



## 파이토치로 심층 신경망 객체 생성

import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 100),
    nn.ReLU(),
    nn.Linear(100, 10)
)


# 구조 확인 패키지 (torchinfo) 설치 및 실행

!pip install torchinfo
from torchinfo import summary

summary(model, input_size = (32, 28, 28))


# 파이토치 심층 신경망 객체를 GPU에 적제
# keras 객체는 이 과정을 자동 수행하나, 파이토치 모델은 직접 수행해야 함.

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


# 손실 함수와 옵티마이저 객체 생성

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
# CrossEntropy 클래스 => 소프트맥스 + 크로스 엔트로피 함수 병합
# model.parameters => 옵티마이저가 최적화할 모든 모델 파라미터 반환



## 파이토치 심층 신경망 훈련 알고리즘 구현(fit() 메서드 X, 직접 코딩)

epochs = 5
batches = int(len(train_scaled) / 32)

for epoch in range(epochs): # 에포크 반복
  model.train()
  train_loss = 0 # 에포크 손실 초기화

  for batch in range(batches): # 배치 반복
    inputs = train_scaled[batch * 32:(batch + 1)*32].to(device)
    targets = train_target[batch * 32:(batch + 1)*32].to(device) # 배치 입력&타깃 준비
    optimizer.zero_grad() # 옵티마이저 그레디언트 초기화
    outputs = model(inputs) # 입력 전달 to 모델
    loss = criterion(outputs, targets) # 모델 출력 & 타깃에 따라 손실 계산
    loss.backward() # 손실 역전파
    optimizer.step() # 모델 파라미터 업데이트
    train_loss += loss.item() # 에포크 손실 기록(추가)
  print(f"에포크:{epoch + 1}, 손실: {train_loss/batches: .4f}") # 에포크 손실 출력



## 검증 정확도 산출 알고리즘

model.eval()
with torch.no_grad():
  val_scaled = val_scaled.to(device)
  val_target = val_target.to(device)
  outputs = model(val_scaled)
  predicts = torch.argmax(outputs, 1)
  correct = (predicts == val_target).sum().item()


accuracy = correct / len(val_target)
print(f"검증 정확도: {accuracy:.4f}")
